In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2
import os
import random
import numpy as np
import shutil

# Path Sumber (Yang isinya hanya file asli .0.jpg)
path_sumber = '/content/drive/MyDrive/uas ml/sukulen'

# Path Tujuan (Folder baru agar data asli tidak kotor)
path_tujuan = '/content/drive/MyDrive/uas ml/sukulen_augmented'

folders = ['daun_runcing', 'daun_bulat']
target_jumlah = 1550
ukuran_target = (224, 224)

# [Fungsi augmentasi_variasi(img) sama persis dengan kodemu sebelumnya]
def augmentasi_variasi(img):
    if img is None:
        return None
    pilihan = random.choice(['flip', 'rotasi', 'crop_scale', 'color_jitter'])
    if pilihan == 'flip':
        flip_code = random.choice([0, 1, -1])
        return cv2.flip(img, flip_code)
    elif pilihan == 'rotasi':
        angle = random.randint(-25, 25)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)
    elif pilihan == 'crop_scale':
        h, w = img.shape[:2]
        crop_factor = random.uniform(0.8, 0.9)
        new_h, new_w = int(h * crop_factor), int(w * crop_factor)
        y_start = random.randint(0, h - new_h)
        x_start = random.randint(0, w - new_w)
        cropped_img = img[y_start:y_start+new_h, x_start:x_start+new_w]
        return cv2.resize(cropped_img, ukuran_target)
    elif pilihan == 'color_jitter':
        alpha = random.uniform(0.8, 1.2)
        beta = random.randint(-30, 30)
        return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

# 1. Membuat folder tujuan utama jika belum ada
if not os.path.exists(path_tujuan):
    os.makedirs(path_tujuan)

print("Memulai Augmentasi ke Folder Baru...")

for folder in folders:
    p_sumber = os.path.join(path_sumber, folder)
    p_tujuan = os.path.join(path_tujuan, folder)

    # 2. Membuat subfolder (daun_runcing/bulat) di folder tujuan
    if not os.path.exists(p_tujuan):
        os.makedirs(p_tujuan)

    files_awal = sorted([f for f in os.listdir(p_sumber) if f.endswith(('.jpg', '.png', '.jpeg'))])
    print(f"\nMemproses kelas {folder}...")

    temp_files_asli = []

    # 3. Kopi (Copy) file asli ke folder tujuan
    for nama_lama in files_awal:
        path_lama = os.path.join(p_sumber, nama_lama)
        img_asli = cv2.imread(path_lama)
        if img_asli is None:
            continue

        img_resized = cv2.resize(img_asli, ukuran_target)
        path_baru_tujuan = os.path.join(p_tujuan, nama_lama)

        cv2.imwrite(path_baru_tujuan, img_resized)
        temp_files_asli.append(nama_lama)

    jumlah_saat_ini = len(temp_files_asli)
    tracker_varian = {f: 0 for f in temp_files_asli}

    print(f"Berhasil mengopi {jumlah_saat_ini} foto asli. Menambah {target_jumlah - jumlah_saat_ini} foto augmentasi...")

    # 4. Augmentasi disimpen langsung ke p_tujuan
    while jumlah_saat_ini < target_jumlah:
        file_induk = random.choice(temp_files_asli)
        id_tanaman = file_induk.split('.')[0]

        # Ambil gambar induknya dari folder tujuan (yang sudah di-resize tadi)
        path_file = os.path.join(p_tujuan, file_induk)
        img = cv2.imread(path_file)

        if img is not None:
            img_aug = augmentasi_variasi(img)

            if img_aug is not None:
                tracker_varian[file_induk] += 1
                nama_aug = f"{id_tanaman}.{tracker_varian[file_induk]}.jpg"

                while os.path.exists(os.path.join(p_tujuan, nama_aug)):
                    tracker_varian[file_induk] += 1
                    nama_aug = f"{id_tanaman}.{tracker_varian[file_induk]}.jpg"

                cv2.imwrite(os.path.join(p_tujuan, nama_aug), img_aug)
                jumlah_saat_ini += 1

print("\n Dataset baru yang utuh ada di folder:", path_tujuan)

Memulai Augmentasi ke Folder Baru...

Memproses kelas daun_runcing...
Berhasil mengopi 317 foto asli. Menambah 1233 foto augmentasi...

Memproses kelas daun_bulat...
Berhasil mengopi 303 foto asli. Menambah 1247 foto augmentasi...

 Dataset baru yang utuh ada di folder: /content/drive/MyDrive/uas ml/sukulen_augmented


DATA SPLITTING

In [ ]:
# DATA SPLIT
import tensorflow as tf
import matplotlib.pyplot as plt

# PATH
path_dataset = '/content/drive/MyDrive/uas ml/sukulen_augmented'

batch_size = 32
img_size = (224, 224)

print("--- Memuat Data Training ---")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    path_dataset,
    validation_split=0.2, # Menyisihkan 20% awal untuk Val/Test
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

# AMBIL NAMA KELAS DI SINI (Sebelum dataset di-prefetch)
class_names = train_dataset.class_names

print("\n--- Memuat Data Validation & Testing ---")
val_test_dataset = tf.keras.utils.image_dataset_from_directory(
    path_dataset,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

# Memecah sisa 20% tadi menjadi dua: 10% untuk Validation, 10% untuk Testing
val_batches = tf.data.experimental.cardinality(val_test_dataset)
val_dataset = val_test_dataset.take(val_batches // 2)
test_dataset = val_test_dataset.skip(val_batches // 2)

# Mengoptimalkan performa pemuatan data agar proses training ngebut
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

print(f"\n=== RINGKASAN DISTRIBUSI DATASET ===")
print(f"Kelas yang terdeteksi: {class_names}")

--- Memuat Data Training ---
Found 3100 files belonging to 2 classes.
Using 2480 files for training.

--- Memuat Data Validation & Testing ---
Found 3100 files belonging to 2 classes.
Using 620 files for validation.

=== RINGKASAN DISTRIBUSI DATASET ===
Kelas yang terdeteksi: ['daun_bulat', 'daun_runcing']
